# CTG: exploração dos dados

Só o conjunto de treino (1755 exames, 282 registros).

**Ordem do notebook**
1. Dados e classes (células 1 e 2)
2. Medidas por classe: boxplots e fração de zeros (3 a 5)
3. Correlação entre as medidas (6 e 7)
4. Padronização, PCA e K-means (8 a 11)
5. Assimetria e histogramas (12)

**Convenções**
- Classes: 0 normal, 1 suspeito, 2 patológico (NSP - 1).
- As contagens (AC, FM, UC, DL, DS, DP) foram convertidas em taxa por segundo na preparação (scripts/prepare_ctg).
- Todas as figuras e tabelas saem em `results/` com o prefixo `ctg_`.
- PCA e K-means recebem os dados padronizados, porque são sensíveis à escala.

In [ ]:
# Célula 1 - imports e dados

import matplotlib.pyplot as plt
import numpy as np
from statstech.cluster import KMeans
from statstech.decomposition import PCA
from statstech.preprocessing import StandardScaler

from analysis.config import SEED, STYLE
from analysis.io import load_dataset
from analysis.report import save_figure, save_table

plt.style.use(STYLE)

CLASS_NAMES = ["Normal", "Suspeito", "Patológico"]
CLASS_COLORS = ["tab:blue", "tab:orange", "tab:red"]

d = load_dataset("ctg")
X, y, names = d["X_train"], d["y_train"], d["feature_names"]  # só o treino
idx = {name: i for i, name in enumerate(names)}
print(X.shape, np.bincount(y))  # esperado: (1755, 20) [1363 245 147]

In [ ]:
# Célula 2 - gráfico de barras com contagem de exames por classe
# gera (ctg_classes.png)

counts = np.bincount(y)
percentages = counts / counts.sum() * 100
labels = [
    f"{count} ({pct:.1f}%)".replace(".", ",") for count, pct in zip(counts, percentages)
]

fig, ax = plt.subplots()
bars = ax.bar(CLASS_NAMES, counts, color=CLASS_COLORS)
ax.bar_label(bars, labels=labels)
ax.set_ylim(0, counts.max() * 1.12)
ax.set_ylabel("Número de exames")
ax.set_title("Exames por classe (treino)")
save_figure(fig, "ctg_classes")
plt.show()

**Leitura.** 1363 exames normais (77,7%), 245 suspeitos (14,0%) e 147 patológicos (8,4%).
Prever sempre "normal" já acertaria 78%, então a acurácia sozinha engana. Para avaliar os modelos, usar F1 macro e o recall do patológico.

In [ ]:
# Célula 3 - boxplots das medidas por classe
# Gera (ctg_boxplots.png)

UNITS = {
    "LB": "bpm",
    "Min": "bpm",
    "Max": "bpm",
    "Mode": "bpm",
    "Mean": "bpm",
    "Median": "bpm",
    "Variance": "bpm²",
    "AC": "1/s",
    "FM": "1/s",
    "UC": "1/s",
    "DL": "1/s",
    "DS": "1/s",
    "DP": "1/s",
    "ASTV": "% do tempo",
    "ALTV": "% do tempo",
}

fig, axes = plt.subplots(4, 5, figsize=(16, 12))
for j, ax in enumerate(axes.flat):
    values_by_class = [X[y == k, j] for k in range(3)]
    box = ax.boxplot(values_by_class, patch_artist=True)
    for patch, color in zip(box["boxes"], CLASS_COLORS):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    unit = UNITS.get(names[j], "")
    ax.set_title(f"{names[j]} ({unit})" if unit else names[j])
    ax.set_xticks([1, 2, 3], ["N", "S", "P"])

fig.suptitle("Medidas por classe (treino). N: normal, S: suspeito, P: patológico")
fig.tight_layout()
save_figure(fig, "ctg_boxplots")
plt.show()

In [ ]:
# Célula 4 - medidas que separam as classes
# Mediana de cada medida por classe. Confere numericamente o que os boxplots
# da célula 3 mostram.

medians = np.array([np.median(X[y == k], axis=0) for k in range(3)])  # (3, 20)

print(f"{'medida':<10}" + "".join(f"{c:>12}" for c in CLASS_NAMES))
for j, name in enumerate(names):
    print(f"{name:<10}" + "".join(f"{medians[k, j]:>12.4g}" for k in range(3)))

## Medidas que separam as classes

- **ASTV**: a mediana do normal fica em torno de 42%, e as do suspeito e do patológico em torno de 63% e 65%.
- **AC**: o normal tem mediana acima de zero; suspeito e patológico ficam colados em zero.
- **DP**: separa o patológico. A mediana dele é maior que zero, e nas outras duas classes é zero.
- **Mean, Median, Mode, MLTV**: o patológico fica mais baixo, mas as caixas são largas e há sobreposição.
- **Variance**: o suspeito tem valores muito baixos e o patológico, os mais altos.
- **Tendency**: o patológico concentra valores negativos; normal e suspeito, de 0 a 1.
- **Separam pouco**: LB, Max, Nmax e UC (as caixas das três classes se sobrepõem).

O normal contra o suspeito é a separação mais difícil.

In [ ]:
# Célula 5 - fração de zeros por classe
# gera (ctg_fracao_zeros.csv)

# Boxplots de DS, DP, FM e Nzeros ficam colapsados em zero. Esta tabela mostra
# a fração de zeros por classe e confirma a variância nula do DS no suspeito.
rows = []
for name in ["FM", "DS", "DP", "Nzeros"]:
    zeros = [(X[y == k, idx[name]] == 0).mean() for k in range(3)]
    rows.append({"medida": name, **dict(zip(CLASS_NAMES, np.round(zeros, 3)))})
save_table(rows, "ctg_fracao_zeros")

ds = X[:, idx["DS"]]
for k, class_name in enumerate(CLASS_NAMES):
    n_nonzero = int((ds[y == k] != 0).sum())
    print(
        f"DS, {class_name}: {n_nonzero} linhas não nulas, "
        f"variância {ds[y == k].var():.2e}"
    )

**Por que esta tabela existe.** DS, DP, FM e Nzeros ficam colapsados em zero nos boxplots.

- **DS** só tem 2 exames não nulos em 1755 (1 normal e 1 patológico) e nenhum no suspeito. Na classe suspeita a variância é exatamente 0, então a covariância dela fica singular no QDA.
- **DP** é zero em 96% dos normais e 95% dos suspeitos, mas só em 41% dos patológicos.
- **FM** é zero em 67% dos normais, 54% dos suspeitos e 48% dos patológicos.
- **Nzeros** é quase igual nas três classes (74% a 85%).

In [ ]:
# Célula 6 - matriz de correlação
# gera (ctg_correlacao.png)

corr = np.corrcoef(X, rowvar=False)
assert corr.shape == (len(names), len(names))
assert not np.isnan(corr).any()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(names)), names, rotation=90)
ax.set_yticks(range(len(names)), names)
fig.colorbar(im, ax=ax, label="Correlação de Pearson")
ax.set_title("Correlação entre as medidas (treino)")
save_figure(fig, "ctg_correlacao")
plt.show()

In [ ]:
# Célula 7 - pares muito correlacionados

i, j = np.triu_indices_from(corr, k=1)
strong = np.abs(corr[i, j]) > 0.7
for a, b, c in zip(i[strong], j[strong], corr[i, j][strong]):
    print(f"{names[a]} x {names[b]}: {c:.2f}")

## Pares muito correlacionados (|r| > 0,7)

Só há um bloco: **LB, Mode, Mean e Median**.

- Mean x Median: 0,96; Mode x Median: 0,94; Mode x Mean: 0,90
- LB x Median: 0,79; LB x Mean: 0,74; LB x Mode: 0,73

Essas medidas dizem quase a mesma coisa. Na interpretação, os coeficientes individuais delas ficam instáveis (o efeito se divide entre elas), e interpretá-las separadamente engana. DS não tem correlação relevante com nada.

In [ ]:
# Célula 8 - padronização dos dados
# alimenta o PCA e o KMeans, que são sensíveis à escala das variáveis

# O PCA só centraliza e o K-means usa distância euclidiana: sem padronizar,
# Variance (em bpm², valores até ~200) dominaria as taxas (da ordem de 0.001).
# O scaler é ajustado só no treino, que é tudo o que este notebook usa.

X_scaled = StandardScaler().fit_transform(X)
print(X_scaled.shape)  # esperado: (1755, 20)

In [ ]:
# Célula 9 - PCA 2D
# gera (ctg_pca_2d.png)

pca_2d = PCA(n_components=2)
Z = pca_2d.fit_transform(X_scaled)
print(Z.shape)  # esperado: (1755, 2)

fig, ax = plt.subplots(figsize=(7, 6))
for k in range(3):
    ax.scatter(
        Z[y == k, 0],
        Z[y == k, 1],
        s=12,
        alpha=0.5,
        color=CLASS_COLORS[k],
        label=CLASS_NAMES[k],
    )
ratio = pca_2d.explained_variance_ratio_
ax.set_xlabel(f"Componente 1 ({ratio[0]:.1%} da variância)")
ax.set_ylabel(f"Componente 2 ({ratio[1]:.1%} da variância)")
ax.set_title(f"PCA 2D das medidas (treino), {ratio.sum():.1%} da variância")
ax.legend()
save_figure(fig, "ctg_pca_2d")
plt.show()

**Leitura do PCA 2D.** As duas primeiras componentes explicam 43,6% da variância (27,6% e 16,0%).
Os patológicos se espalham para valores negativos da componente 1, e os suspeitos se concentram à direita. Os normais ocupam o centro e se misturam com os dois lados.
A separação existe nas pontas, mas há muita sobreposição, principalmente entre normal e suspeito.

In [ ]:
# Célula 10 - variância acumulada do PCA
# gera (ctg_pca_variancia.png)

pca_full = PCA()  # todas as componentes
pca_full.fit(X_scaled)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
print(len(cumulative), cumulative[-1])  # esperado: 20 e 1.0

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
ax.axhline(0.9, linestyle="--", color="gray")
ax.set_xlabel("Número de componentes")
ax.set_ylabel("Variância explicada acumulada")
ax.set_title("Variância acumulada do PCA (treino)")
save_figure(fig, "ctg_pca_variancia")
plt.show()

**Leitura da variância acumulada.** Precisamos de 11 componentes para reter 90% da variância. Não há um "cotovelo" claro: a informação está espalhada por muitas medidas.

In [ ]:
# Célula 11 - KMeans x classe
# gera (ctg_kmeans_classe.csv)

# fit_predict devolve o grupo (0, 1 ou 2) de cada exame, sem usar y.
# A numeração dos grupos é arbitrária: só as contagens da tabela importam.
# np.add.at soma 1 em table[grupo, classe] para cada exame e acumula
# corretamente quando o mesmo par se repete.

kmeans = KMeans(n_clusters=3, random_state=SEED)
clusters = kmeans.fit_predict(X_scaled)
print(np.bincount(clusters))  # tamanho de cada grupo

table = np.zeros((3, 3), dtype=int)  # linha = grupo, coluna = classe
np.add.at(table, (clusters, y), 1)
assert table.sum() == len(y)
assert (table.sum(axis=0) == np.bincount(y)).all()  # colunas = tamanho das classes

rows = [{"grupo": g, **dict(zip(CLASS_NAMES, table[g].tolist()))} for g in range(3)]
save_table(rows, "ctg_kmeans_classe")
print(table)

## K-means (k = 3) contra as classes

| | Normal | Suspeito | Patológico |
|---|---|---|---|
| Grupo 0 (170) | 68 | 9 | 93 |
| Grupo 1 (680) | 413 | 217 | 50 |
| Grupo 2 (905) | 882 | 19 | 4 |

Sem usar os rótulos:
- O grupo 0 é o mais "patológico": 55% dele é patológico (a base tem 8,4%) e ele reúne 63% de todos os patológicos.
- O grupo 2 é quase só normal (97%).
- O grupo 1 é a mistura: reúne 89% dos suspeitos, mas também 413 normais.

Os grupos acompanham o diagnóstico em parte, sem alinhar com ele. A numeração dos grupos é arbitrária.

In [ ]:
# Célula 12 - assimetria (skewness) e histograma das medidas
# gera (ctg_histogramas.png)

# Assimetria (skewness) = média de z³, com z = (x - média) / desvio.
# Perto de 0: simétrica. Acima de 1 em módulo: cauda longa.

skewness = np.mean(((X - X.mean(axis=0)) / X.std(axis=0)) ** 3, axis=0)
print(skewness.shape)  # esperado: (20,)

order = np.argsort(-np.abs(skewness))
for j in order:
    print(f"{names[j]}: {skewness[j]:.2f}")

hist_names = ["Variance", "AC", "FM", "UC", "DL", "DS", "DP"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, name in zip(axes.flat, hist_names):
    ax.hist(X[:, idx[name]], bins=40)
    ax.set_title(name)
    ax.set_ylabel("Exames")
for ax in axes.flat[len(hist_names) :]:
    ax.set_visible(False)
fig.tight_layout()
save_figure(fig, "ctg_histogramas")
plt.show()

## Medidas assimétricas (|assimetria| > 1)

DS (29,97), FM (8,86), DP (4,07), Nzeros (3,58), Variance (2,67), ALTV (2,27), MSTV (1,65), DL (1,64), MLTV (1,45), AC (1,18).

**Para a comparação dos modelos (transformação log1p):**
- Variance, ALTV, MSTV e MLTV são candidatas reais.
- As taxas (FM, DP, DL, AC) têm um pico em zero e valores da ordem de 0,001: o log1p quase não muda nada nelas, a menos que sejam reescaladas antes.
- DS: a assimetria vem de 2 exames não nulos, e nenhuma transformação resolve. A remoção da coluna é uma decisão do grupo.